# Post-Training Validation Inspection

This notebook reuses the repository training and decoding helpers to inspect a trained checkpoint in two ways:

1. Synthetic center-gap sliding windows with `Gapped Input`, `Ground Truth`, and `Prediction` shown side by side.
2. Natural annotated-gap windows from the gapped song, where only `Gapped Input` and `Prediction` are available.


In [ ]:
from pathlib import Path
import json
import sys
from typing import Any, Dict, Iterable, List, Mapping, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython import get_ipython
from IPython.display import Audio, Markdown, display

REPO_ROOT = Path("..").resolve()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from audio_infill.train import (
    Trainer,
    compute_log_spectrogram_data,
    compute_validation_crop_bounds,
    frame_bounds_to_sample_bounds,
    parse_args as train_parse_args,
    save_waveform,
    slugify_component,
)

ip = get_ipython()
if ip is not None:
    ip.run_line_magic("matplotlib", "inline")


In [ ]:
# Edit only this cell before running the notebook.
EXPERIMENT: Dict[str, Any] = {
    "config_path": REPO_ROOT / "configs/train/multigap_decoded_loss_val_v2.yaml",
    "ds_dir": REPO_ROOT / "data/processed/multigap",
    "sample": "wav_test_multigap_4x_1p0s_2p0s_5p0s_10p0s",
    "wav_path": None,
    "checkpoint": REPO_ROOT / "outputs/runs/longrun_multigap_decoded_loss_val_v2/checkpoints/latest.pt",
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
    "mask_len": None,
    "crop_context_frames": None,
    "stride_frames": None,
    "max_windows": 4,
    "inpaint_iters": 1,
    "skip_annotated_gap_windows": True,
    "analyze_natural_gaps": True,
    "max_natural_gaps": None,
    "export_results": False,
    "export_root": REPO_ROOT / "outputs/debug/post_training_validation_inspection",
    "run_name": "post_training_validation_inspection",
}

assert Path(EXPERIMENT["config_path"]).exists(), EXPERIMENT["config_path"]
assert Path(EXPERIMENT["checkpoint"]).exists(), EXPERIMENT["checkpoint"]
display(Markdown("## Experiment"))
display(EXPERIMENT)


In [ ]:
def resolve_source_wav(repo_root: Path, ann: Optional[Mapping[str, Any]], fallback_wav: Path) -> Path:
    if ann is None:
        return fallback_wav.resolve()
    candidates: List[Path] = []
    raw = ann.get("source_wav")
    if raw:
        candidates.append(Path(str(raw)))
        candidates.append(repo_root / Path(str(raw)).name)
        candidates.append(repo_root / "data/raw" / Path(str(raw)).name)
    generated = ann.get("generated_wav")
    if generated:
        stem = Path(str(generated)).stem
        if "_gap_" in stem:
            base = stem.split("_gap_")[0]
        elif "_multigap_" in stem:
            base = stem.split("_multigap_")[0]
        else:
            base = stem
        candidates.append(repo_root / "data/raw" / f"{base}.wav")
    candidates.append(fallback_wav)
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f"Could not resolve source wav. Tried: {candidates}")


cfg_args = [
    "--config", str(EXPERIMENT["config_path"]),
    "--device", str(EXPERIMENT["device"]),
    "--inpaint-only",
]
if EXPERIMENT.get("sample"):
    cfg_args.extend([
        "--ds-dir", str(EXPERIMENT["ds_dir"]),
        "--sample", str(EXPERIMENT["sample"]),
        "--auto-hparam",
    ])
elif EXPERIMENT.get("wav_path"):
    cfg_args.extend(["--wav-path", str(EXPERIMENT["wav_path"])])

cfg, _ = train_parse_args(cfg_args)
cfg.output_dir = str(REPO_ROOT / "outputs/debug")
cfg.run_name = str(EXPERIMENT["run_name"])
cfg.resume = None
cfg.inpaint_only = True
cfg.validation_every = 0
cfg.test_fill_every = 0

trainer = Trainer(cfg)
trainer.load_checkpoint(str(EXPERIMENT["checkpoint"]))
trainer.model.eval()

annotation = getattr(cfg, "_annotation", None)
source_wav_path = resolve_source_wav(REPO_ROOT, annotation, Path(cfg.wav_path))
source_data = trainer._load_audio_sample(
    wav_path=str(source_wav_path),
    target_sr=cfg.target_sr,
    gap_start_s=cfg.gap_start_s,
    gap_end_s=cfg.gap_end_s,
    ann=annotation,
)
source_codes = source_data["codes"]
source_wav = source_data["wav"].squeeze(0).numpy()
annotated_gaps = source_data["gaps_f"]

display(Markdown("## Loaded trainer and source audio"))
display({
    "resolved_source_wav": str(source_wav_path),
    "frames": int(source_codes.shape[1]),
    "seq_len": int(cfg.seq_len),
    "mask_len_default": int(cfg.mask_len_max),
    "annotated_gaps": annotated_gaps,
})


In [ ]:
def window_overlaps_ranges(start: int, span_len: int, ranges: Iterable[Tuple[int, int]]) -> bool:
    end = start + span_len
    for r0, r1 in ranges:
        if start < r1 and end > r0:
            return True
    return False


def iter_window_starts(total_frames: int, seq_len: int, stride_frames: int, blocked_ranges: Iterable[Tuple[int, int]] = ()):
    max_start = max(0, int(total_frames) - int(seq_len))
    starts = list(range(0, max_start + 1, max(1, int(stride_frames))))
    if starts and starts[-1] != max_start:
        starts.append(max_start)
    for start in starts:
        if blocked_ranges and window_overlaps_ranges(start, seq_len, blocked_ranges):
            continue
        yield int(start)


def compute_context_window(total_frames: int, g0: int, g1: int) -> Tuple[int, int, int, int]:
    gap_len = int(g1 - g0)
    if cfg.ctx_left is not None and cfg.ctx_right is not None:
        ctx_left = int(cfg.ctx_left)
        ctx_right = int(cfg.ctx_right)
    else:
        budget = max(0, int(cfg.max_len) - gap_len - 16)
        ctx_left = budget // 2
        ctx_right = budget - ctx_left
    left = max(0, int(g0) - ctx_left)
    right = min(int(total_frames), int(g1) + ctx_right)
    local_g0 = int(g0) - left
    local_g1 = int(g1) - left
    return left, right, local_g0, local_g1


def make_audio_gap_proxy(audio: np.ndarray, mask_sample_bounds: Tuple[int, int]) -> np.ndarray:
    out = np.asarray(audio, dtype=np.float32).copy()
    s0, s1 = mask_sample_bounds
    s0 = max(0, min(int(s0), max(0, out.shape[0] - 1)))
    s1 = max(s0 + 1, min(int(s1), out.shape[0]))
    out[s0:s1] = 0.0
    return out


def make_multi_spectrogram_figure(
    audio_items: Sequence[Tuple[str, np.ndarray]],
    sr: int,
    title_prefix: str,
    time_offset_s: float = 0.0,
    min_freq: float = 30.0,
):
    prepared = []
    vmax = None
    for label, audio in audio_items:
        times, freqs, spec_db = compute_log_spectrogram_data(audio, sr)
        prepared.append((label, times, freqs, spec_db))
        local_max = float(np.max(spec_db))
        vmax = local_max if vmax is None else max(vmax, local_max)
    vmin = float(vmax) - 80.0
    fig, axes = plt.subplots(1, len(prepared), figsize=(6.2 * len(prepared), 4.8), sharex=True, sharey=True, constrained_layout=True)
    if len(prepared) == 1:
        axes = [axes]
    mesh = None
    for ax, (label, times, freqs, spec_db) in zip(axes, prepared):
        freq_mask = freqs >= max(float(min_freq), float(freqs[0]) if freqs.size else float(min_freq))
        freqs_plot = freqs[freq_mask]
        spec_plot = spec_db[freq_mask, :]
        mesh = ax.pcolormesh(
            times + float(time_offset_s),
            freqs_plot,
            spec_plot,
            shading="auto",
            cmap="magma",
            vmin=vmin,
            vmax=float(vmax),
        )
        ax.set_yscale("log")
        ax.set_ylim(max(float(min_freq), float(freqs_plot[0])), float(freqs_plot[-1]))
        ax.set_title(f"{title_prefix} | {label}")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Frequency (Hz)")
    if mesh is not None:
        fig.colorbar(mesh, ax=axes, label="dB")
    return fig


def make_multi_waveform_figure(
    audio_items: Sequence[Tuple[str, np.ndarray]],
    sr: int,
    title_prefix: str,
    mask_sample_bounds: Tuple[int, int],
    crop_sample_bounds: Tuple[int, int],
    time_offset_s: float = 0.0,
):
    fig, axes = plt.subplots(2, len(audio_items), figsize=(5.5 * len(audio_items), 6.5), sharey="row", constrained_layout=True)
    if len(audio_items) == 1:
        axes = np.asarray(axes).reshape(2, 1)
    crop_start, crop_end = crop_sample_bounds
    mask_start, mask_end = mask_sample_bounds
    for col, (label, audio) in enumerate(audio_items):
        arr = np.asarray(audio, dtype=np.float32).reshape(-1)
        times = (np.arange(arr.shape[0], dtype=np.float32) / float(sr)) + float(time_offset_s)
        axes[0, col].plot(times, arr, linewidth=1.2)
        axes[0, col].axvspan(
            (mask_start / float(sr)) + float(time_offset_s),
            (mask_end / float(sr)) + float(time_offset_s),
            color="tab:red",
            alpha=0.12,
        )
        axes[0, col].set_title(f"{title_prefix} | {label} | Full")
        axes[0, col].set_xlabel("Time (s)")
        axes[0, col].set_ylabel("Amplitude")

        crop = arr[crop_start:crop_end]
        crop_times = (np.arange(crop.shape[0], dtype=np.float32) / float(sr)) + float(time_offset_s) + (crop_start / float(sr))
        axes[1, col].plot(crop_times, crop, linewidth=1.2)
        axes[1, col].axvspan(
            (max(mask_start, crop_start) / float(sr)) + float(time_offset_s),
            (min(mask_end, crop_end) / float(sr)) + float(time_offset_s),
            color="tab:red",
            alpha=0.12,
        )
        axes[1, col].set_title(f"{title_prefix} | {label} | Crop")
        axes[1, col].set_xlabel("Time (s)")
        axes[1, col].set_ylabel("Amplitude")
    return fig


def export_result(result: Dict[str, Any], export_root: Path) -> Path:
    example_id = slugify_component(result["example_id"])
    out_dir = export_root / example_id
    out_dir.mkdir(parents=True, exist_ok=True)
    bundle = {
        "analysis_kind": result["analysis_kind"],
        "example_id": result["example_id"],
        "window_start": int(result["window_start"]),
        "window_end": int(result["window_end"]),
        "mask_start": int(result["mask_start"]),
        "mask_end": int(result["mask_end"]),
        "mask_len": int(result["mask_len"]),
        "crop_frame_bounds": tuple(int(v) for v in result["crop_frame_bounds"]),
        "crop_sample_bounds": tuple(int(v) for v in result["crop_sample_bounds"]),
        "mask_sample_bounds": tuple(int(v) for v in result["mask_sample_bounds"]),
        "x_masked_codes": result["x_masked_codes"].clone(),
        "input_codes": result["input_codes"].clone(),
        "pred_filled_codes": result["pred_filled_codes"].clone(),
        "gapped_audio": np.asarray(result["gapped_audio"], dtype=np.float32),
        "pred_audio": np.asarray(result["pred_audio"], dtype=np.float32),
    }
    if result.get("y_target_codes") is not None:
        bundle["y_target_codes"] = result["y_target_codes"].clone()
    if result.get("target_audio") is not None:
        bundle["target_audio"] = np.asarray(result["target_audio"], dtype=np.float32)
    torch.save(bundle, out_dir / "bundle.pt")

    save_waveform(out_dir / "gapped_window.wav", result["gapped_audio"], cfg.target_sr)
    save_waveform(out_dir / "pred_window.wav", result["pred_audio"], cfg.target_sr)
    crop_start, crop_end = result["crop_sample_bounds"]
    save_waveform(out_dir / "gapped_gap_crop.wav", result["gapped_audio"][crop_start:crop_end], cfg.target_sr)
    save_waveform(out_dir / "pred_gap_crop.wav", result["pred_audio"][crop_start:crop_end], cfg.target_sr)
    files = {
        "bundle": "bundle.pt",
        "gapped_window": "gapped_window.wav",
        "pred_window": "pred_window.wav",
        "gapped_gap_crop": "gapped_gap_crop.wav",
        "pred_gap_crop": "pred_gap_crop.wav",
    }
    if result.get("target_audio") is not None:
        save_waveform(out_dir / "target_window.wav", result["target_audio"], cfg.target_sr)
        save_waveform(out_dir / "target_gap_crop.wav", result["target_audio"][crop_start:crop_end], cfg.target_sr)
        files["target_window"] = "target_window.wav"
        files["target_gap_crop"] = "target_gap_crop.wav"
    metadata = {
        "analysis_kind": result["analysis_kind"],
        "example_id": result["example_id"],
        "window_start": int(result["window_start"]),
        "window_end": int(result["window_end"]),
        "mask_start": int(result["mask_start"]),
        "mask_end": int(result["mask_end"]),
        "mask_len": int(result["mask_len"]),
        "crop_frame_bounds": [int(v) for v in result["crop_frame_bounds"]],
        "crop_sample_bounds": [int(v) for v in result["crop_sample_bounds"]],
        "files": files,
    }
    with open(out_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)
    return out_dir


def display_result(result: Dict[str, Any]) -> None:
    crop_start, crop_end = result["crop_sample_bounds"]
    if result.get("target_audio") is not None:
        audio_items = [
            ("Gapped Input", result["gapped_audio"]),
            ("Ground Truth", result["target_audio"]),
            ("Prediction", result["pred_audio"]),
        ]
    else:
        audio_items = [
            ("Gapped Input", result["gapped_audio"]),
            ("Prediction", result["pred_audio"]),
        ]
    title = result["title"]
    spec_full = make_multi_spectrogram_figure(
        audio_items,
        sr=cfg.target_sr,
        title_prefix=title,
        time_offset_s=float(result["window_start"]) / float(trainer.encoder.frame_rate),
    )
    spec_crop = make_multi_spectrogram_figure(
        [(label, audio[crop_start:crop_end]) for label, audio in audio_items],
        sr=cfg.target_sr,
        title_prefix=f"{title} | Crop",
        time_offset_s=float(result["window_start"] + result["crop_frame_bounds"][0]) / float(trainer.encoder.frame_rate),
    )
    wave_fig = make_multi_waveform_figure(
        audio_items,
        sr=cfg.target_sr,
        title_prefix=title,
        mask_sample_bounds=result["mask_sample_bounds"],
        crop_sample_bounds=result["crop_sample_bounds"],
        time_offset_s=float(result["window_start"]) / float(trainer.encoder.frame_rate),
    )
    display(Markdown(f"### {title}"))
    display(spec_full)
    display(spec_crop)
    display(wave_fig)
    plt.close(spec_full)
    plt.close(spec_crop)
    plt.close(wave_fig)
    for label, audio in audio_items:
        display(Markdown(f"**{label} audio**"))
        display(Audio(audio, rate=cfg.target_sr))


def run_center_gap_infill(start_frame: int) -> Dict[str, Any]:
    seq_len = int(cfg.seq_len)
    mask_len = int(EXPERIMENT["mask_len"] or cfg.mask_len_max)
    mask_start = int((seq_len - mask_len) // 2)
    mask_end = int(mask_start + mask_len)
    target_codes = source_codes[:, start_frame : start_frame + seq_len].clone()
    x_masked_codes = target_codes.clone()
    x_masked_codes[:, mask_start:mask_end] = trainer.mask_token
    xb = x_masked_codes.unsqueeze(0).to(trainer.device)
    for _ in range(max(1, int(EXPERIMENT["inpaint_iters"]))):
        logits = trainer.model(xb)
        pred = logits.argmax(dim=-1)
        xb[0, :, mask_start:mask_end] = pred[0, :, mask_start:mask_end]
    pred_codes = xb[0].detach().cpu()
    target_audio = trainer._decode_validation_window_audio(target_codes)
    total_samples = target_audio.shape[0]
    crop_frame_bounds = compute_validation_crop_bounds(seq_len, mask_start, mask_len, EXPERIMENT["crop_context_frames"])
    crop_sample_bounds = frame_bounds_to_sample_bounds(total_samples, seq_len, crop_frame_bounds[0], crop_frame_bounds[1])
    mask_sample_bounds = frame_bounds_to_sample_bounds(total_samples, seq_len, mask_start, mask_end)
    gapped_audio = make_audio_gap_proxy(target_audio, mask_sample_bounds)
    pred_audio = trainer._decode_validation_window_audio(pred_codes)
    return {
        "analysis_kind": "synthetic_center_gap",
        "example_id": f"synthetic_window_{int(start_frame):06d}",
        "title": (
            f"Synthetic window={int(start_frame)}:{int(start_frame + seq_len)} | "
            f"mask={mask_start}:{mask_end}"
        ),
        "window_start": int(start_frame),
        "window_end": int(start_frame + seq_len),
        "mask_start": int(mask_start),
        "mask_end": int(mask_end),
        "mask_len": int(mask_len),
        "crop_frame_bounds": tuple(int(v) for v in crop_frame_bounds),
        "crop_sample_bounds": tuple(int(v) for v in crop_sample_bounds),
        "mask_sample_bounds": tuple(int(v) for v in mask_sample_bounds),
        "x_masked_codes": x_masked_codes,
        "input_codes": x_masked_codes.clone(),
        "y_target_codes": target_codes,
        "pred_filled_codes": pred_codes,
        "gapped_audio": gapped_audio,
        "target_audio": target_audio,
        "pred_audio": pred_audio,
    }


def run_natural_gap_infill(gap_index: int, g0: int, g1: int) -> Dict[str, Any]:
    left, right, local_g0, local_g1 = compute_context_window(trainer.frames, int(g0), int(g1))
    input_codes = trainer.codes[:, left:right].clone()
    x_masked_codes = input_codes.clone()
    x_masked_codes[:, local_g0:local_g1] = trainer.mask_token
    xb = x_masked_codes.unsqueeze(0).to(trainer.device)
    for _ in range(max(1, int(EXPERIMENT["inpaint_iters"]))):
        logits = trainer.model(xb)
        pred = logits.argmax(dim=-1)
        xb[0, :, local_g0:local_g1] = pred[0, :, local_g0:local_g1]
    pred_codes = xb[0].detach().cpu()
    gapped_audio = trainer._decode_validation_window_audio(input_codes)
    pred_audio = trainer._decode_validation_window_audio(pred_codes)
    seq_len = int(input_codes.shape[1])
    total_samples = min(gapped_audio.shape[0], pred_audio.shape[0])
    crop_frame_bounds = compute_validation_crop_bounds(seq_len, local_g0, int(g1 - g0), EXPERIMENT["crop_context_frames"])
    crop_sample_bounds = frame_bounds_to_sample_bounds(total_samples, seq_len, crop_frame_bounds[0], crop_frame_bounds[1])
    mask_sample_bounds = frame_bounds_to_sample_bounds(total_samples, seq_len, local_g0, local_g1)
    return {
        "analysis_kind": "natural_gap",
        "example_id": f"natural_gap_{int(gap_index):02d}",
        "title": (
            f"Natural gap #{int(gap_index)} | window={left}:{right} | "
            f"gap={local_g0}:{local_g1}"
        ),
        "window_start": int(left),
        "window_end": int(right),
        "mask_start": int(local_g0),
        "mask_end": int(local_g1),
        "mask_len": int(g1 - g0),
        "crop_frame_bounds": tuple(int(v) for v in crop_frame_bounds),
        "crop_sample_bounds": tuple(int(v) for v in crop_sample_bounds),
        "mask_sample_bounds": tuple(int(v) for v in mask_sample_bounds),
        "x_masked_codes": x_masked_codes,
        "input_codes": input_codes,
        "y_target_codes": None,
        "pred_filled_codes": pred_codes,
        "gapped_audio": gapped_audio,
        "target_audio": None,
        "pred_audio": pred_audio,
    }


In [ ]:
mask_len = int(EXPERIMENT["mask_len"] or cfg.mask_len_max)
stride_frames = int(EXPERIMENT["stride_frames"] or max(1, mask_len // 2))
blocked_ranges = annotated_gaps if EXPERIMENT["skip_annotated_gap_windows"] else []
starts = list(iter_window_starts(source_codes.shape[1], cfg.seq_len, stride_frames, blocked_ranges=blocked_ranges))
if EXPERIMENT["max_windows"] is not None:
    starts = starts[: int(EXPERIMENT["max_windows"])]

display(Markdown(f"## Synthetic center-gap windows: {len(starts)}"))
synthetic_results: List[Dict[str, Any]] = []
for start in starts:
    result = run_center_gap_infill(int(start))
    synthetic_results.append(result)
    display_result(result)
    if EXPERIMENT["export_results"]:
        out_dir = export_result(result, Path(EXPERIMENT["export_root"]) / "synthetic")
        display(Markdown(f"Saved artifacts to `{out_dir}`"))

display(Markdown(f"Finished synthetic analysis. Collected {len(synthetic_results)} windows."))


In [ ]:
if EXPERIMENT["analyze_natural_gaps"]:
    gap_entries = list(enumerate(trainer.gaps_f))
    if EXPERIMENT["max_natural_gaps"] is not None:
        gap_entries = gap_entries[: int(EXPERIMENT["max_natural_gaps"])]
    display(Markdown(f"## Natural annotated gaps: {len(gap_entries)}"))
    natural_results: List[Dict[str, Any]] = []
    for gap_index, (g0, g1) in gap_entries:
        result = run_natural_gap_infill(int(gap_index), int(g0), int(g1))
        natural_results.append(result)
        display_result(result)
        if EXPERIMENT["export_results"]:
            out_dir = export_result(result, Path(EXPERIMENT["export_root"]) / "natural")
            display(Markdown(f"Saved artifacts to `{out_dir}`"))
    display(Markdown(f"Finished natural-gap analysis. Collected {len(natural_results)} windows."))
else:
    display(Markdown("## Natural annotated gaps analysis disabled"))
